# PatchCore ViT-B/16 One-Layer Defect-Tuning (`x224`)

This notebook is the canonical training and evaluation workflow for the one-layer ViT-B/16 PatchCore experiment with defect-aware threshold tuning.

Run this notebook from top to bottom to reproduce the experiment locally. The notebook prepares the data split, trains the model when needed, evaluates the trained model, and writes outputs into the experiment-local artifact folders.

Artifacts written by this notebook:
- `[experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/checkpoints](experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/checkpoints)` for model checkpoints
- `[experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/plots](experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/plots)` for saved figures
- `[experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/results](experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/results)` for metrics, score files, and CSV outputs


## Setup

This cell installs or checks optional notebook dependencies before the rest of the workflow runs.

In [ ]:
# -- 0. Install dependencies if needed -----------------------------------------
import importlib
import importlib.util
import subprocess
import sys
for pkg in ['timm', 'tqdm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('timm + tqdm ready')



## Imports

This cell imports the Python packages used across training, evaluation, plotting, and artifact export.

In [ ]:
# -- 1. Imports ----------------------------------------------------------------
import os, gc, json, random, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

print('Device:', DEVICE)
if USE_CUDA:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')

from IPython.display import display



## Configuration

This cell resolves the repo root, defines experiment settings, and points all outputs to the local artifact folders.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# ?? 2. Configuration ??????????????????????????????????????????????????????????
DATA_PATH  = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
IMAGE_SIZE = 224

TRAIN_NORMAL_N = 40_000
TUNE_NORMAL_N  =  5_000
TUNE_DEFECT_N  =   5_000
TEST_NORMAL_N  =  5_000
TEST_DEFECT_N  =   250

VIT_FEATURE_BLOCK = 6
PATCH_EMBED_DIM   = 256
MEMORY_BANK_MAX_PATCHES = 600_000
SCORE_CHUNK      = 512
PATCHCORE_NN_K   = 3
TOPK_PATCH_RATIO = 0.03

BATCH_SIZE  = 128
NUM_WORKERS = 0

THRESHOLD_PERCENTILE_MIN   = 90
THRESHOLD_PERCENTILE_MAX   = 99.9
THRESHOLD_PERCENTILE_STEPS = 100
THRESHOLD_GRID_STEPS       = 300

FORCE_REBUILD_SCORES = False
FORCE_RERUN_UMAP = False

ARTIFACT_DIR = str(PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts')
CHECKPOINTS_DIR = os.path.join(ARTIFACT_DIR, 'checkpoints')
PLOTS_DIR = os.path.join(ARTIFACT_DIR, 'plots')
RESULTS_DIR = os.path.join(ARTIFACT_DIR, 'results')
MODEL_EXPORT_PATH = os.path.join(CHECKPOINTS_DIR, 'patchcore_vit_b16_model.pt')
METRICS_EXPORT_PATH = os.path.join(RESULTS_DIR, 'evaluation_metrics.json')
SUMMARY_EXPORT_PATH = os.path.join(RESULTS_DIR, 'summary.json')
SCORES_EXPORT_PATH = os.path.join(RESULTS_DIR, 'scores.npz')
UMAP_PNG_PATH = os.path.join(PLOTS_DIR, 'umap_test_embeddings.png')
UMAP_CSV_PATH = os.path.join(RESULTS_DIR, 'umap_test_embeddings.csv')
for path_str in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
    os.makedirs(path_str, exist_ok=True)

# Align config with a saved checkpoint when reusing artifacts.
LEGACY_CHECKPOINT_PATH = os.path.join(ARTIFACT_DIR, 'patchcore_vit_b16_model.pt')
if os.path.exists(MODEL_EXPORT_PATH) or os.path.exists(LEGACY_CHECKPOINT_PATH):
    checkpoint_probe_path = MODEL_EXPORT_PATH if os.path.exists(MODEL_EXPORT_PATH) else LEGACY_CHECKPOINT_PATH
    try:
        checkpoint_probe = torch.load(checkpoint_probe_path, map_location='cpu')
        ckpt_cfg = checkpoint_probe.get('config', {}) if isinstance(checkpoint_probe, dict) else {}
        if 'patch_embed_dim' in ckpt_cfg:
            PATCH_EMBED_DIM = int(ckpt_cfg['patch_embed_dim'])
        if 'vit_feature_block' in ckpt_cfg:
            VIT_FEATURE_BLOCK = int(ckpt_cfg['vit_feature_block'])
        print(f'Aligned notebook config to saved checkpoint: block={VIT_FEATURE_BLOCK}, embed_dim={PATCH_EMBED_DIM}')
    except Exception as exc:
        print(f'Warning: could not probe saved checkpoint config: {exc}')

print('Artifacts will be saved to:', ARTIFACT_DIR)

print(f'ViT block={VIT_FEATURE_BLOCK}  embed_dim={PATCH_EMBED_DIM}')
print(f'Bank cap={MEMORY_BANK_MAX_PATCHES:,}  batch={BATCH_SIZE}  workers={NUM_WORKERS}')
print(f'Test defect ratio: {TEST_DEFECT_N / TEST_NORMAL_N * 100:.1f}%  ({TEST_DEFECT_N} / {TEST_NORMAL_N})')
print(f'FORCE_REBUILD_SCORES={FORCE_REBUILD_SCORES}  FORCE_RERUN_UMAP={FORCE_RERUN_UMAP}')




## Load Dataset

This cell loads the legacy WM-811K pickle, cleans labels, and prepares the dataframe used for the experiment split.

In [ ]:
# ── 3. Load & clean dataset ───────────────────────────────────────────────────
# Only DataFrames are created here — NO tensors, NO float32 copies.
# Raw wafer maps stay as object-dtype numpy arrays in the DataFrame.

from wafer_defect.data.legacy_pickle import read_legacy_pickle

df = read_legacy_pickle(DATA_PATH)
print('Raw shape:', df.shape)

def parse_failure_label(v):
    if v is None: return 'unknown'
    if isinstance(v, float) and np.isnan(v): return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)

df = df.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()

invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)

normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()

print(f'Labeled: {len(df):,}   Normal: {len(normal_df):,}   Defect: {len(defect_df):,}')
print('\nDefect breakdown:')
print(defect_df['failure_label'].value_counts())

## ── 4. Split ──────────────────────────────────────────────────────────────────

This cell continues the experiment workflow and writes its outputs into the local artifact folders.

In [ ]:
# ── 4. Split ──────────────────────────────────────────────────────────────────
req_n = TRAIN_NORMAL_N + TUNE_NORMAL_N + TEST_NORMAL_N
req_d = TUNE_DEFECT_N + TEST_DEFECT_N

if len(normal_df) < req_n:
    raise ValueError(f'Need {req_n:,} normals, have {len(normal_df):,}')
if len(defect_df) < req_d:
    raise ValueError(f'Need {req_d:,} defects, have {len(defect_df):,}')

rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)

a = TRAIN_NORMAL_N
b = a + TUNE_NORMAL_N
c = b + TEST_NORMAL_N

train_normal_df = ns.iloc[0:a].copy()
tune_normal_df  = ns.iloc[a:b].copy()
test_normal_df  = ns.iloc[b:c].copy()
tune_defect_df  = ds.iloc[0:TUNE_DEFECT_N].copy()
test_defect_df  = ds.iloc[TUNE_DEFECT_N:TUNE_DEFECT_N + TEST_DEFECT_N].copy()

# Free the large unsplit dataframes — keep only the slices
del df, normal_df, defect_df, ns, ds
gc.collect()

print(f'Train normal : {len(train_normal_df):>7,}')
print(f'Tune  normal : {len(tune_normal_df):>7,}')
print(f'Tune  defect : {len(tune_defect_df):>7,}')
print(f'Test  normal : {len(test_normal_df):>7,}')
print(f'Test  defect : {len(test_defect_df):>7,}')

## ── 5. Lazy WaferDataset ──────────────────────────────────────────────────────

This cell continues the experiment workflow and writes its outputs into the local artifact folders.

In [ ]:
# ── 5. Lazy WaferDataset ──────────────────────────────────────────────────────
#
# KEY RAM FIX: no tensor is created until __getitem__ is called for a single
# sample. The DataLoader batches these calls across NUM_WORKERS processes,
# so at most BATCH_SIZE tensors exist in memory at once.
#
# Raw map storage: 1 wafer map ≈ 26×26 int8 ≈ 700 bytes
# vs tensor:       1 wafer map ≈ 224×224×3 float32 ≈ 602 KB
# Ratio: ~860× more compact in the DataFrame.

class WaferDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, size: int = 224):
        self.maps   = frame['waferMap'].values   # numpy object array of 2-D arrays
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size   = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x   = torch.tensor(arr, dtype=torch.long)
        x   = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()  # [3, H, W]
        x   = F.interpolate(
                  x.unsqueeze(0),
                  size=(self.size, self.size),
                  mode='nearest'
              ).squeeze(0)                                            # [3, 224, 224]
        return x, int(self.labels[idx])


loader_kw = dict(
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_CUDA,
    persistent_workers=(NUM_WORKERS > 0),
)

train_loader       = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
tune_normal_loader = DataLoader(WaferDataset(tune_normal_df,  IMAGE_SIZE), **loader_kw)
tune_defect_loader = DataLoader(WaferDataset(tune_defect_df,  IMAGE_SIZE), **loader_kw)
test_normal_loader = DataLoader(WaferDataset(test_normal_df,  IMAGE_SIZE), **loader_kw)
test_defect_loader = DataLoader(WaferDataset(test_defect_df,  IMAGE_SIZE), **loader_kw)

for name, ldr in [('train_normal', train_loader), ('tune_normal', tune_normal_loader),
                  ('tune_defect', tune_defect_loader), ('test_normal', test_normal_loader),
                  ('test_defect', test_defect_loader)]:
    print(f'{name:<14}: {len(ldr):>4} batches')

# Smoke-test one batch
xb, yb = next(iter(train_loader))
print(f'\nBatch shape: {tuple(xb.shape)}  dtype={xb.dtype}  sample labels={yb[:4].tolist()}')

## Define Model

This cell defines the PatchCore feature extractor and supporting model components for this branch.

In [ ]:
# ── 6. ViT-B/16 feature extractor ────────────────────────────────────────────
# Forward hook on block[VIT_FEATURE_BLOCK] captures [B, 197, 768] token output.
# We drop the CLS token → [B, 196, 768] spatial patch tokens.
# Projected to [B, 196, PATCH_EMBED_DIM] for memory efficiency.

class ViTPatchExtractor(nn.Module):
    def __init__(self, block_idx: int = VIT_FEATURE_BLOCK,
                 proj_dim: int = PATCH_EMBED_DIM):
        super().__init__()
        self.vit = timm.create_model(
            'vit_base_patch16_224.augreg_in21k_ft_in1k',
            pretrained=True,
            num_classes=0,
        )
        self._feat = None
        self.vit.blocks[block_idx].register_forward_hook(
            lambda m, i, o: setattr(self, '_feat', o)
        )
        self.proj = nn.Linear(768, proj_dim, bias=False)

    def forward(self, x):
        self.vit(x)                    # hook fires mid-pass
        return self._feat[:, 1:, :]    # drop CLS → [B, 196, 768]


extractor = ViTPatchExtractor().to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False

# Smoke-test
with torch.inference_mode():
    dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out   = extractor(dummy)
    proj  = extractor.proj(out)
print(f'ViT block-{VIT_FEATURE_BLOCK} output : {tuple(out.shape)}')   # [2, 196, 768]
print(f'After projection         : {tuple(proj.shape)}')              # [2, 196, 128]

## Train and Score

This cell builds the PatchCore memory bank, computes anomaly scores, and saves the score bundle needed for later evaluation.

In [ ]:
# -- 7. Build PatchCore memory bank --------------------------------------------
# Build the memory bank only when saved local score artifacts are unavailable or stale.

REQUIRED_SCORE_KEYS = {
    'train_scores_z',
    'tune_normal_scores_z',
    'tune_defect_scores_z',
    'test_normal_scores_z',
    'test_defect_scores_z',
    'train_mu',
    'train_std',
}
PARTIAL_SCORE_KEYS = {
    'train_scores_z',
    'tune_normal_scores_z',
    'test_normal_scores_z',
    'test_defect_scores_z',
    'train_mu',
    'train_std',
}

HAS_SAVED_CHECKPOINT = os.path.exists(MODEL_EXPORT_PATH)
checkpoint_artifact = None
if HAS_SAVED_CHECKPOINT:
    checkpoint_artifact = torch.load(MODEL_EXPORT_PATH, map_location='cpu')
    extractor.load_state_dict(checkpoint_artifact['extractor_state_dict'])
    extractor = extractor.to(DEVICE).eval()
    print(f'Loaded saved checkpoint: {MODEL_EXPORT_PATH}')

saved_score_keys = set()
if os.path.exists(SCORES_EXPORT_PATH):
    with np.load(SCORES_EXPORT_PATH) as existing_scores:
        saved_score_keys = set(existing_scores.files)

HAS_FULL_SAVED_SCORES = REQUIRED_SCORE_KEYS.issubset(saved_score_keys)
HAS_PARTIAL_SAVED_SCORES = PARTIAL_SCORE_KEYS.issubset(saved_score_keys)
REUSE_SAVED_RESULTS = HAS_SAVED_CHECKPOINT and HAS_FULL_SAVED_SCORES and not FORCE_REBUILD_SCORES
REUSE_PARTIAL_SAVED_RESULTS = HAS_SAVED_CHECKPOINT and HAS_PARTIAL_SAVED_SCORES and not FORCE_REBUILD_SCORES

memory_bank = None
memory_bank_t = None

if REUSE_SAVED_RESULTS or REUSE_PARTIAL_SAVED_RESULTS:
    print('Skipping memory bank rebuild and reusing saved local artifacts.')
else:
    def extract_embeddings(xb: torch.Tensor) -> torch.Tensor:
        """L2-normalised embeddings: [B*196, proj_dim] on GPU."""
        with torch.inference_mode():
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)
                emb  = extractor.proj(feat)
            emb = emb.float().reshape(-1, emb.shape[-1])
            emb = F.normalize(emb, p=2, dim=1)
        return emb

    sampled, total_seen, sample_ratio = [], 0, None

    print('Building memory bank...')
    bank_iter = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc='Bank build',
        unit='batch',
    )
    for step, (xb, _) in bank_iter:
        xb  = xb.to(DEVICE)
        emb = extract_embeddings(xb)
        total_seen += len(emb)

        if sample_ratio is None:
            tokens_per_img  = len(emb) // len(xb)
            estimated_total = tokens_per_img * len(train_normal_df)
            sample_ratio    = min(1.0, MEMORY_BANK_MAX_PATCHES / estimated_total)
            print(f'  Tokens/image : {tokens_per_img}')
            print(f'  Est. total   : {estimated_total:,}')
            print(f'  Sample ratio : {sample_ratio:.5f}')

        if sample_ratio < 1.0:
            k   = max(1, int(round(len(emb) * sample_ratio)))
            idx = torch.randperm(len(emb), device=DEVICE)[:k]
            emb = emb[idx]

        sampled.append(emb)

        if (step + 1) % 20 == 0 or (step + 1) == len(train_loader):
            n = sum(len(e) for e in sampled)
            bank_iter.set_postfix(bank_tokens=f'{n:,}')

    memory_bank = torch.cat(sampled, dim=0)
    del sampled; gc.collect()

    if len(memory_bank) > MEMORY_BANK_MAX_PATCHES:
        idx = torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX_PATCHES]
        memory_bank = memory_bank[idx]

    memory_bank   = F.normalize(memory_bank, p=2, dim=1).contiguous()
    memory_bank_t = memory_bank.t().contiguous()

    mb_mb = memory_bank.element_size() * memory_bank.numel() / 1e6
    print(f'Final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d  ({mb_mb:.1f} MB VRAM)')



## Reload Scores

This cell reloads the saved score bundle so threshold selection and downstream evaluation use the persisted results.

In [ ]:
# -- 8. PatchCore scoring -------------------------------------------------------

def min_dist_to_bank(patches, bank_t, chunk=512, k=3):
    out = []
    for i in range(0, len(patches), chunk):
        p    = patches[i:i+chunk]
        sim  = p @ bank_t
        kk   = min(k, sim.shape[1])
        best = sim.topk(kk, dim=1).values
        dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best, min=0.0)).mean(dim=1)
        out.append(dist)
    return torch.cat(out)


def score_loader(loader, bank_t, topk_ratio=0.1, nn_k=3, desc='Scoring'):
    scores = []
    with torch.inference_mode():
        pbar = tqdm(loader, total=len(loader), desc=desc, unit='batch')
        for xb, _ in pbar:
            xb = xb.to(DEVICE)
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)
                emb  = extractor.proj(feat)
            emb = emb.float().reshape(-1, emb.shape[-1])
            emb = F.normalize(emb, p=2, dim=1)

            ps   = min_dist_to_bank(emb, bank_t, SCORE_CHUNK, nn_k)
            b    = len(xb)
            ps   = ps.reshape(b, -1)
            topk = max(1, int(round(ps.shape[1] * topk_ratio)))
            topk = min(topk, ps.shape[1])
            s    = ps.topk(topk, dim=1).values.mean(dim=1)
            scores.append(s.cpu())
    return torch.cat(scores).numpy()

kw = dict(topk_ratio=TOPK_PATCH_RATIO, nn_k=PATCHCORE_NN_K)

if REUSE_SAVED_RESULTS or REUSE_PARTIAL_SAVED_RESULTS:
    with np.load(SCORES_EXPORT_PATH) as d:
        train_scores_z       = d['train_scores_z']
        tune_normal_scores_z = d['tune_normal_scores_z']
        tune_defect_scores_z = d['tune_defect_scores_z'] if 'tune_defect_scores_z' in d.files else None
        test_normal_scores_z = d['test_normal_scores_z']
        test_defect_scores_z = d['test_defect_scores_z']
        mu  = float(d['train_mu'])
        std = float(d['train_std'])
    print(f'Loaded saved score artifacts from: {SCORES_EXPORT_PATH}')
else:
    train_scores       = score_loader(train_loader,       memory_bank_t, desc='Score train-normal', **kw)
    tune_normal_scores = score_loader(tune_normal_loader, memory_bank_t, desc='Score tune-normal', **kw)
    tune_defect_scores = score_loader(tune_defect_loader, memory_bank_t, desc='Score tune-defect', **kw)
    test_normal_scores = score_loader(test_normal_loader, memory_bank_t, desc='Score test-normal', **kw)
    test_defect_scores = score_loader(test_defect_loader, memory_bank_t, desc='Score test-defect', **kw)

    mu  = float(np.mean(train_scores))
    std = float(np.std(train_scores) + 1e-8)
    def znorm(x): return (x - mu) / std

    train_scores_z       = znorm(train_scores)
    tune_normal_scores_z = znorm(tune_normal_scores)
    tune_defect_scores_z = znorm(tune_defect_scores)
    test_normal_scores_z = znorm(test_normal_scores)
    test_defect_scores_z = znorm(test_defect_scores)

    np.savez_compressed(
        SCORES_EXPORT_PATH,
        train_scores_z=train_scores_z,
        tune_normal_scores_z=tune_normal_scores_z,
        tune_defect_scores_z=tune_defect_scores_z,
        test_normal_scores_z=test_normal_scores_z,
        test_defect_scores_z=test_defect_scores_z,
        train_mu=np.array(mu), train_std=np.array(std),
    )
    print(f'Scores saved. mu={mu:.6f}  sigma={std:.6f}')




## Reload Scores

This cell reloads the saved score bundle so threshold selection and downstream evaluation use the persisted results.

In [ ]:
# -- 9. Threshold selection (labeled tune split) -------------------------------
# Use labeled tune data when available. Otherwise fall back to the saved threshold
# stored in the checkpoint or evaluation metrics.

def _save_threshold_plot(normal_scores, defect_scores, threshold_value, title, out_path):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(normal_scores, bins=50, alpha=0.55, label='Tune normal', color='steelblue', density=True)
    if defect_scores is not None:
        ax.hist(defect_scores, bins=50, alpha=0.45, label='Tune defect', color='tomato', density=True)
    ax.axvline(threshold_value, color='red', linewidth=2, linestyle='--', label=f'Threshold z={threshold_value:.3f}')
    ax.set_xlabel('Anomaly score (z)')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.show()

if tune_defect_scores_z is not None:
    y_tune = np.concatenate([
        np.zeros(len(tune_normal_scores_z), dtype=int),
        np.ones(len(tune_defect_scores_z), dtype=int),
    ])
    score_tune = np.concatenate([tune_normal_scores_z, tune_defect_scores_z])

    grid_lo = float(np.percentile(score_tune, 1.0))
    grid_hi = float(np.percentile(score_tune, 99.0))
    threshold_grid = np.linspace(grid_lo, grid_hi, THRESHOLD_GRID_STEPS)

    best_bal_acc = -1.0
    best_f1 = -1.0
    threshold_z = float(np.median(score_tune))
    for t in threshold_grid:
        pred = (score_tune > t).astype(int)
        tp = int(((pred == 1) & (y_tune == 1)).sum())
        tn = int(((pred == 0) & (y_tune == 0)).sum())
        fp = int(((pred == 1) & (y_tune == 0)).sum())
        fn = int(((pred == 0) & (y_tune == 1)).sum())

        tpr = tp / max(tp + fn, 1)
        tnr = tn / max(tn + fp, 1)
        bal_acc = 0.5 * (tpr + tnr)
        precision = tp / max(tp + fp, 1)
        recall = tpr
        f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)

        if (bal_acc > best_bal_acc) or (np.isclose(bal_acc, best_bal_acc) and f1 > best_f1):
            best_bal_acc = bal_acc
            best_f1 = f1
            threshold_z = float(t)

    threshold_raw = mu + threshold_z * std
    print(f'Selected z-threshold: {threshold_z:.4f}  raw: {threshold_raw:.6f}')
    print(f'Tune balanced-accuracy: {best_bal_acc:.4f}  tune F1: {best_f1:.4f}')
    _save_threshold_plot(
        tune_normal_scores_z,
        tune_defect_scores_z,
        threshold_z,
        'Score Distribution ? Labeled Tune Split',
        os.path.join(PLOTS_DIR, 'threshold_selection.png'),
    )
else:
    print('Saved scores are missing tune-defect values. Falling back to the stored threshold from the saved checkpoint/results.')
    fallback_metrics = None
    if os.path.exists(METRICS_EXPORT_PATH):
        fallback_metrics = json.loads(Path(METRICS_EXPORT_PATH).read_text(encoding='utf-8'))
    if checkpoint_artifact is not None and 'threshold_z' in checkpoint_artifact:
        threshold_z = float(checkpoint_artifact['threshold_z'])
        threshold_raw = float(checkpoint_artifact.get('threshold_raw', mu + threshold_z * std))
    elif fallback_metrics is not None and 'threshold_z' in fallback_metrics:
        threshold_z = float(fallback_metrics['threshold_z'])
        threshold_raw = float(fallback_metrics.get('threshold_raw', mu + threshold_z * std))
    else:
        raise RuntimeError('No saved threshold is available. Re-run with FORCE_REBUILD_SCORES=True to regenerate the full scoring artifacts.')
    print(f'Loaded saved z-threshold: {threshold_z:.4f}  raw: {threshold_raw:.6f}')
    _save_threshold_plot(
        tune_normal_scores_z,
        None,
        threshold_z,
        'Score Distribution ? Saved Threshold Fallback',
        os.path.join(PLOTS_DIR, 'threshold_selection.png'),
    )



## Evaluate

This cell computes the final evaluation metrics on the held-out split using the selected decision threshold.

In [ ]:
# ?? 10. Final test evaluation ?????????????????????????????????????????????????
# threshold_z = 1.5   # ? uncomment to override

y_true = np.concatenate([np.zeros(len(test_normal_scores_z), dtype=int),
                          np.ones( len(test_defect_scores_z), dtype=int)])
scores = np.concatenate([test_normal_scores_z, test_defect_scores_z])
y_pred = (scores > threshold_z).astype(int)

roc_auc = float(roc_auc_score(y_true, scores))
print(f'ROC-AUC : {roc_auc:.4f}')
print(f'Z-thr   : {threshold_z:.4f}   raw: {threshold_raw:.6f}')
print(classification_report(y_true, y_pred, target_names=['normal','anomaly']))

cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['pred normal','pred anomaly'],
            yticklabels=['true normal','true anomaly'])
axes[0].set_title('Confusion Matrix (Test)')

sns.kdeplot(test_normal_scores_z, label='Normal', fill=True, alpha=0.35, ax=axes[1])
sns.kdeplot(test_defect_scores_z, label='Defect', fill=True, alpha=0.35, ax=axes[1])
axes[1].axvline(threshold_z, color='red', ls='--', label=f'z={threshold_z:.2f}')
axes[1].set_xlabel('Anomaly score (z)'); axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution (Test)'); axes[1].legend()
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, 'confusion_and_score_distribution.png'), dpi=180, bbox_inches='tight')
plt.show()

fig_cm, ax_cm = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax_cm,
            xticklabels=['pred normal','pred anomaly'],
            yticklabels=['true normal','true anomaly'])
ax_cm.set_title('Confusion Matrix (Test)')
plt.tight_layout()
fig_cm.savefig(os.path.join(PLOTS_DIR, 'confusion_matrix.png'), dpi=180, bbox_inches='tight')
plt.show()

fig_dist, ax_dist = plt.subplots(figsize=(8, 4))
sns.kdeplot(test_normal_scores_z, label='Normal', fill=True, alpha=0.35, ax=ax_dist)
sns.kdeplot(test_defect_scores_z, label='Defect', fill=True, alpha=0.35, ax=ax_dist)
ax_dist.axvline(threshold_z, color='red', ls='--', label=f'z={threshold_z:.2f}')
ax_dist.set_xlabel('Anomaly score (z)')
ax_dist.set_ylabel('Density')
ax_dist.set_title('Score Distribution (Test)')
ax_dist.legend()
plt.tight_layout()
fig_dist.savefig(os.path.join(PLOTS_DIR, 'score_distribution.png'), dpi=180, bbox_inches='tight')
plt.show()

# Per-class defect recall
tmp = test_defect_df.copy()
tmp['score']    = test_defect_scores_z
tmp['detected'] = (test_defect_scores_z > threshold_z).astype(int)
print('Per-class defect recall:')
defect_recall_df = (
    tmp.groupby('failure_label')
       .agg(count=('detected','count'), detected=('detected','sum'),
            recall=('detected','mean'), mean_score=('score','mean'))
       .round(3).sort_values('recall')
       .reset_index()
)
display(defect_recall_df)
defect_recall_df.to_csv(os.path.join(RESULTS_DIR, 'defect_recall.csv'), index=False)




## UMAP Visualization

This cell projects saved embeddings for qualitative visualization and saves the resulting UMAP outputs.

In [ ]:
# -- 11. UMAP diagnostic (test split) -------------------------------------------
# Visualize image-level embeddings (mean pooled patch embeddings).

import sys, subprocess
from sklearn.decomposition import PCA

if (not FORCE_RERUN_UMAP) and os.path.exists(UMAP_PNG_PATH) and os.path.exists(UMAP_CSV_PATH):
    print(f'Using saved UMAP artifacts: {UMAP_PNG_PATH} and {UMAP_CSV_PATH}')
else:
    if 'test_normal_loader' not in globals() or 'test_defect_loader' not in globals():
        print('Recreating test loaders for UMAP...')
        _loader_kw = dict(
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=USE_CUDA,
            persistent_workers=(NUM_WORKERS > 0),
        )
        test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **_loader_kw)
        test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **_loader_kw)

    try:
        import umap.umap_ as umap
    except Exception:
        print('Installing umap-learn...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
        import umap.umap_ as umap

    MAX_UMAP_IMAGES_PER_SPLIT = 2000

    def collect_image_embeddings(loader, max_images=2000, desc='embed'):
        embs, labels = [], []
        seen = 0
        with torch.inference_mode():
            for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
                xb = xb.to(DEVICE)
                with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                    feat = extractor(xb)
                    emb  = extractor.proj(feat)
                img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
                embs.append(img_emb.cpu().numpy())
                labels.append(yb.cpu().numpy())
                seen += len(yb)
                if seen >= max_images:
                    break

        X = np.concatenate(embs, axis=0)[:max_images]
        y = np.concatenate(labels, axis=0)[:max_images]
        return X, y

    print('Collecting test embeddings for UMAP...')
    Xn, yn = collect_image_embeddings(test_normal_loader, max_images=MAX_UMAP_IMAGES_PER_SPLIT, desc='Embed test-normal')
    Xd, yd = collect_image_embeddings(test_defect_loader, max_images=MAX_UMAP_IMAGES_PER_SPLIT, desc='Embed test-defect')

    X = np.concatenate([Xn, Xd], axis=0)
    y = np.concatenate([yn, yd], axis=0).astype(int)

    n_pca = min(32, X.shape[1], X.shape[0] - 1)
    Xp = PCA(n_components=n_pca, random_state=SEED).fit_transform(X)

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric='cosine',
        random_state=SEED,
        transform_seed=SEED,
        low_memory=True,
        verbose=True,
     )
    coords = reducer.fit_transform(Xp)

    fig, ax = plt.subplots(figsize=(8, 6))
    m0 = (y == 0)
    m1 = (y == 1)
    ax.scatter(coords[m0, 0], coords[m0, 1], s=8, alpha=0.45, label='Normal', c='steelblue', linewidths=0)
    ax.scatter(coords[m1, 0], coords[m1, 1], s=8, alpha=0.60, label='Defect', c='tomato', linewidths=0)
    ax.set_title('UMAP of Test Image Embeddings (ViT PatchCore)')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend()
    plt.tight_layout()
    fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
    plt.show()

    pd.DataFrame({'umap_1': coords[:, 0], 'umap_2': coords[:, 1], 'label': y}).to_csv(UMAP_CSV_PATH, index=False)
    print(f'Saved UMAP figure: {UMAP_PNG_PATH}')
    print(f'Saved UMAP coords: {UMAP_CSV_PATH}')



## Save Outputs

This cell exports the trained model artifact, metrics, and any final bookkeeping files for reproducibility.

In [ ]:
# ?? 12. Save artifacts and summary ???????????????????????????????????????????
artifact = {
    'extractor_state_dict': extractor.state_dict(),
    'threshold_z'  : float(threshold_z),
    'threshold_raw': float(threshold_raw),
    'train_score_mu' : float(mu),
    'train_score_std': float(std),
    'config': dict(
        backbone='vit_base_patch16_224.augreg_in21k_ft_in1k',
        vit_feature_block=VIT_FEATURE_BLOCK,
        patch_embed_dim=PATCH_EMBED_DIM,
        image_size=IMAGE_SIZE,
        train_normal_n=TRAIN_NORMAL_N,
        tune_normal_n=TUNE_NORMAL_N,
        tune_defect_n=TUNE_DEFECT_N,
        test_normal_n=TEST_NORMAL_N,
        test_defect_n=TEST_DEFECT_N,
        memory_bank_max=MEMORY_BANK_MAX_PATCHES,
        score_chunk=SCORE_CHUNK,
        patchcore_nn_k=PATCHCORE_NN_K,
        topk_patch_ratio=TOPK_PATCH_RATIO,
    ),
}
torch.save(artifact, MODEL_EXPORT_PATH)

metrics = dict(
    roc_auc_z=roc_auc,
    threshold_z=float(threshold_z),
    threshold_raw=float(threshold_raw),
    train_score_mu=float(mu),
    train_score_std=float(std),
    confusion_matrix=cm.tolist(),
    n_test_normal=int(len(test_normal_scores_z)),
    n_test_defect=int(len(test_defect_scores_z)),
)
pd.Series(metrics).to_json(METRICS_EXPORT_PATH, indent=2)

summary = {
    'name': 'PatchCore ViT-B/16 One-Layer Defect-Tuning',
    'checkpoint_path': MODEL_EXPORT_PATH,
    'scores_path': SCORES_EXPORT_PATH,
    'metrics_path': METRICS_EXPORT_PATH,
    'umap_csv_path': UMAP_CSV_PATH,
    'umap_plot_path': UMAP_PNG_PATH,
    'force_rebuild_scores': bool(FORCE_REBUILD_SCORES),
    'force_rerun_umap': bool(FORCE_RERUN_UMAP),
    'reused_saved_checkpoint': bool(HAS_SAVED_CHECKPOINT),
    'reused_saved_scores': bool(REUSE_SAVED_RESULTS or REUSE_PARTIAL_SAVED_RESULTS),
    'has_full_saved_scores': bool(HAS_FULL_SAVED_SCORES),
    'has_partial_saved_scores': bool(HAS_PARTIAL_SAVED_SCORES),
    'metrics': metrics,
    'config': artifact['config'],
}
Path(SUMMARY_EXPORT_PATH).write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Model  ?', MODEL_EXPORT_PATH)
print('Metrics?', METRICS_EXPORT_PATH)
print('Summary?', SUMMARY_EXPORT_PATH)



## Evaluate

This cell computes the final evaluation metrics on the held-out split using the selected decision threshold.

In [ ]:
# ── 12. Cleanup ───────────────────────────────────────────────────────────────
for name in [
    'memory_bank', 'memory_bank_t',
    'train_scores', 'tune_normal_scores', 'tune_defect_scores',
    'test_normal_scores', 'test_defect_scores',
    'train_scores_z', 'tune_normal_scores_z', 'tune_defect_scores_z',
    'test_normal_scores_z', 'test_defect_scores_z',
    'scores', 'y_true', 'y_pred',
    'train_loader', 'tune_normal_loader', 'tune_defect_loader',
    'test_normal_loader', 'test_defect_loader',
]:
    if name in globals(): del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
print('Cleared.')